# 01 – EDA, tiền xử lý, BoW / TF-IDF

Phụ trách: Thanh Hằng. Notebook chạy từ thư mục gốc repo hoặc từ `notebooks/`.

Thứ tự: (1) EDA dữ liệu thô → (2) tiền xử lý → (3) EDA sau tiền xử lý → (4) ma trận BoW / TF-IDF → (5) trực quan.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from collections import Counter
from src import config
from src.preprocessing.clean import clean_text
from src.preprocessing.tokenizer import tokenize
from src.preprocessing.stopwords import load_stopwords
from src.preprocessing.build_clean import build_clean, preprocess_text
from src.representation import bow_tfidf
from src.visualization import plots
pd.set_option("display.max_colwidth", 120)

## 1. EDA dữ liệu thô (`articles.csv`)

In [2]:
raw = pd.read_csv(config.ARTICLES_CSV)
raw["n_words"] = raw["content"].str.split().str.len()
print(raw.shape)
raw[["id", "title", "source", "true_label", "n_words"]].head()

(119, 8)


,id,title,source,true_label,n_words
0,doc_0001,Apple gây tranh cãi vì đóng cửa vĩnh viễn diễn đàn hỗ trợ người dùng,dantri,Công nghệ,513
1,doc_0002,Đề xuất mạng lưới cảnh báo lừa đảo trực tuyến tại Việt Nam,dantri,Công nghệ,866
2,doc_0003,"Việt Nam - Hàn Quốc ký 8 thỏa thuận hợp tác AI, bán dẫn",dantri,Công nghệ,712
3,doc_0004,Tại sao AI không thể “suy nghĩ” giống như con người?,dantri,Công nghệ,1477
4,doc_0005,"Sự cố dịch vụ Microsoft kéo dài bất thường, nhiều doanh nghiệp ảnh hưởng",dantri,Công nghệ,563


In [3]:
pd.crosstab(raw["true_label"], raw["source"], margins=True, margins_name="Tổng")

source,dantri,thanhnien,tuoitre,vnexpress,Tổng
true_label,,,,,
Công nghệ,5,5,5,5,20
Giải trí,5,5,5,5,20
Kinh tế,5,5,5,5,20
Sức khỏe,5,5,5,5,20
Thể thao,5,4,5,5,19
Thời sự,5,5,5,5,20
Tổng,30,29,30,30,119


In [4]:
raw.groupby("true_label")["n_words"].describe().round(0).astype(int)

,count,mean,std,min,25%,50%,75%,max
true_label,,,,,,,,
Công nghệ,20,761,453,293,468,626,924,2073
Giải trí,20,782,253,512,634,696,844,1380
Kinh tế,20,782,280,227,632,720,932,1555
Sức khỏe,20,724,373,291,512,652,812,1980
Thể thao,19,616,179,285,464,613,748,956
Thời sự,20,711,592,226,387,520,855,2995


In [5]:
plots.plot_topic_counts(raw["true_label"]);
plots.plot_length_distribution(raw["n_words"], raw["true_label"], name="eda_length_raw", xlabel="Số từ mỗi bài (trước tiền xử lý)");

## 2. Tiền xử lý: làm sạch → tách từ → bỏ stopwords

Ví dụ từng bước trên một đoạn của bài đầu tiên:

In [6]:
sample = raw["content"].iloc[0][:300]
sw = load_stopwords()
step1 = clean_text(sample, lowercase=False)
step2 = tokenize(step1)
step3 = preprocess_text(sample, sw)
print("GỐC     :", sample, "\n")
print("SẠCH    :", step1, "\n")
print("TÁCH TỪ :", " ".join(step2), "\n")
print("SAU STOP:", " ".join(step3))

GỐC     : Apple sẽ đóng cửa vĩnh viễn diễn đàn trực tuyến hỗ trợ người dùng từ tuần sau. Đây là động thái gây tranh cãi vì diễn đàn này vẫn được người dùng tin tưởng để phản ánh các tình trạng lỗi gặp phải.
Nhiều người dùng khi truy cập vào Diễn đàn Cộng đồng Hỗ trợ Apple đã bất ngờ nhận được thông báo cho bi 

SẠCH    : Apple sẽ đóng cửa vĩnh viễn diễn đàn trực tuyến hỗ trợ người dùng từ tuần sau Đây là động thái gây tranh cãi vì diễn đàn này vẫn được người dùng tin tưởng để phản ánh các tình trạng lỗi gặp phải Nhiều người dùng khi truy cập vào Diễn đàn Cộng đồng Hỗ trợ Apple đã bất ngờ nhận được thông báo cho bi 

TÁCH TỪ : Apple sẽ đóng_cửa vĩnh_viễn diễn_đàn trực_tuyến hỗ_trợ người dùng từ tuần sau Đây là động_thái gây tranh_cãi vì diễn_đàn này vẫn được người dùng tin_tưởng để phản_ánh các tình_trạng lỗi gặp phải Nhiều người dùng khi truy_cập vào Diễn_đàn Cộng_đồng Hỗ_trợ Apple đã bất_ngờ nhận được thông_báo cho bi 

SAU STOP: apple đóng_cửa vĩnh_viễn diễn_đàn trực_tuyến hỗ_trợ dùn

In [7]:
df = build_clean()   # ghi data/processed/articles_clean.csv
df["n_tokens"] = df["tokens"].fillna("").str.split().str.len()
df[["true_label", "n_tokens"]].groupby("true_label").describe().round(0).astype(int)

Đọc 119 bài; 293 stopwords; tách từ bằng pyvi


Ghi data/processed/articles_clean.csv: token/bài TB 315 (min 102, max 1131)


n_tokens                                    
              count mean  std  min  25%  50%  75%   max
true_label                                             
Công nghệ        20  305  167  123  193  257  364   742
Giải trí         20  325  104  223  252  289  352   545
Kinh tế          20  349  117  111  294  344  411   659
Sức khỏe         20  334  168  154  222  312  362   922
Thể thao         19  264   80  127  200  271  301   420
Thời sự          20  311  224  102  186  229  404  1131

## 3. EDA sau tiền xử lý

In [8]:
docs = df["tokens"].fillna("").tolist()
cnt = Counter(w for d in docs for w in d.split())
print(f"Tổng token: {sum(cnt.values()):,} | từ vựng: {len(cnt):,} | từ xuất hiện đúng 1 lần: {sum(1 for c in cnt.values() if c == 1):,}")
compound = sum(1 for w in cnt if "_" in w)
print(f"Từ ghép (có '_'): {compound:,} ({compound/len(cnt):.0%} từ vựng)")
pd.DataFrame(cnt.most_common(25), columns=["từ", "số lần"])

Tổng token: 37,501 | từ vựng: 6,207 | từ xuất hiện đúng 1 lần: 2,762
Từ ghép (có '_'): 3,949 (64% từ vựng)


,từ,số lần
0,việt_nam,180
1,nhà,151
2,bệnh_viện,143
3,bộ,134
4,tổ_chức,130
5,hoạt_động,119
6,lớn,117
7,tăng,114
8,cao,110
9,sử_dụng,109


In [9]:
plots.plot_length_distribution(df["n_tokens"], df["true_label"], name="eda_length_tokens", xlabel="Số token mỗi bài (sau tiền xử lý)");

In [10]:
plots.plot_top_words(docs, df["true_label"], n=10);

In [11]:
# Luật Zipf: tần suất giảm rất nhanh theo hạng -> đa số từ hiếm, lý do cần min_df
import matplotlib.pyplot as plt
freqs = np.array(sorted(cnt.values(), reverse=True))
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.loglog(np.arange(1, len(freqs) + 1), freqs, color=plots.PALETTE[0])
ax.set(xlabel="Hạng của từ (log)", ylabel="Tần suất (log)", title="Phân bố tần suất từ (Zipf)")
fig.savefig(config.FIGURES_DIR / "eda_zipf.png"); plt.show()

## 4. Ma trận BoW và TF-IDF

BoW dùng unigram (đầu vào LSA/LDA); TF-IDF dùng ngram (1, 2). Tham số: `min_df=2`, `max_df=0.8` (từ trong `src/config.py`).

In [12]:
X_bow, v_bow = bow_tfidf.build_bow(docs)
X_tfidf, v_tfidf = bow_tfidf.build_tfidf(docs)
bow_tfidf.save_matrix("bow", X_bow, v_bow)
bow_tfidf.save_matrix("tfidf", X_tfidf, v_tfidf)
pd.DataFrame({"BoW": bow_tfidf.matrix_stats(X_bow), "TF-IDF": bow_tfidf.matrix_stats(X_tfidf)})

,BoW,TF-IDF
so_bai,119.00,119.00
so_tu,2823.00,5089.00
nnz,17715.00,23170.00
do_thua_%,94.73,96.17


In [13]:
# Ảnh hưởng của min_df lên kích thước từ vựng (căn cứ chọn min_df)
rows = []
for md in [1, 2, 3, 5]:
    X, _ = bow_tfidf.build_bow(docs, min_df=md)
    rows.append({"min_df": md, "số từ": X.shape[1]})
pd.DataFrame(rows)

,min_df,số từ
0,1,6207
1,2,2823
2,3,1951
3,5,1153


In [14]:
terms = bow_tfidf.top_terms(X_tfidf, v_tfidf, df["true_label"], n=8)
pd.DataFrame({k: ", ".join(t) for k, t in terms.items()}, index=["Top từ TF-IDF"]).T

,Top từ TF-IDF
Công nghệ,"dùng, công_ty, công_nghệ, microsoft, kết_nối, sự_cố, mỹ, nền_tảng"
Giải trí,"diễn_viên, phim, khán_giả, nghệ_sĩ, ca_sĩ, nam, nữ, hát"
Kinh tế,"thuế, giá, thị_trường, doanh_nghiệp, đồng, giao_dịch, nvidia, công_ty"
Sức khỏe,"bác_sĩ, người_bệnh, bệnh_viện, điều_trị, cơ_thể, bệnh_nhân, bệnh, y_tế"
Thể thao,"hlv, trận, đội_tuyển, đội, bóng, trận_đấu, giải, bóng_đá"
Thời sự,"tỉnh, địa_phương, lực_lượng, ubnd, quản_lý, nhà_nước, bộ, quốc_hội"


In [15]:
# Cặp từ (bigram) có trọng số cao nhất: dấu hiệu cụm từ có nghĩa
names = np.array(v_tfidf.get_feature_names_out())
bi = [i for i, t in enumerate(names) if " " in t]
w = np.asarray(X_tfidf[:, bi].sum(axis=0)).ravel()
[(names[bi][i].replace(" ", " + "), round(w[i], 2)) for i in np.argsort(w)[::-1][:12]]

[('tỉ + đồng', np.float64(1.0)),
 ('đội + bóng', np.float64(0.89)),
 ('jensen + huang', np.float64(0.84)),
 ('nhà + đầu_tư', np.float64(0.79)),
 ('trải + nghiệm', np.float64(0.79)),
 ('mạng + xã_hội', np.float64(0.74)),
 ('man + utd', np.float64(0.74)),
 ('đội_tuyển + việt_nam', np.float64(0.69)),
 ('mối + quan_hệ', np.float64(0.68)),
 ('căn + nhà', np.float64(0.67)),
 ('tỷ + usd', np.float64(0.63)),
 ('trí_tuệ + nhân_tạo', np.float64(0.6))]

## 5. Trực quan (nhãn thật)

Chưa có nhãn cụm từ phần phân cụm, nên các hình dưới tô màu theo `true_label`. Khi có nhãn cụm, thay `df["true_label"]` bằng nhãn cụm.

In [16]:
plots.plot_scatter_2d(X_tfidf, df["true_label"], "tsne", "t-SNE – TF-IDF, tô màu theo chủ đề thật", "scatter_tsne_true");

In [17]:
plots.plot_scatter_2d(X_tfidf, df["true_label"], "umap", "UMAP – TF-IDF, tô màu theo chủ đề thật", "scatter_umap_true");

In [18]:
plots.plot_wordcloud_grid(docs, df["true_label"], "wordcloud_true", weights=plots.idf_weights(v_tfidf),
                         title="Wordcloud theo chủ đề thật (tần suất × idf)");

### Minh họa confusion matrix

Chỉ để kiểm tra hàm vẽ: K-Means k=6 trên TF-IDF rút gọn bằng SVD. **Không phải kết quả chính thức** – bảng 6 thí nghiệm do Anh Dũng chạy.

In [19]:
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
Z = TruncatedSVD(50, random_state=config.RANDOM_STATE).fit_transform(X_tfidf)
demo = KMeans(6, n_init=10, random_state=config.RANDOM_STATE).fit_predict(Z)
plots.plot_confusion_matrix(df["true_label"], demo, "cm_demo", title="Minh họa: K-Means k=6 (TF-IDF + SVD)");
print("Purity:", round(plots.purity(df["true_label"], demo), 3))

Purity: 0.597


## Nhận xét

- Dữ liệu cân bằng (19–20 bài/chủ đề), độ dài trung vị khoảng 676 từ thô.
- Sau tiền xử lý còn ~300 token/bài; phần lớn từ vựng chỉ xuất hiện 1 lần → `min_df=2` bỏ hơn một nửa từ vựng (6.207 → 2.823).
- Ma trận rất thưa (> 94%), đúng với văn bản ngắn → cần LSA để giảm chiều trước khi phân cụm.
- Top từ TF-IDF mỗi chủ đề khớp trực giác; Công nghệ và Kinh tế có chung từ (nvidia, công_ty) → dễ lẫn cụm, đã ghi chú ở báo cáo thu thập dữ liệu.